# 03 — User Feature Mart

**입력:** `data/processed/events_clean.csv`, `data/processed/profile_clean.csv`  
**출력:** `data/processed/user_features.csv` (1행 = 1유저, 12,500행)  
**범위:** 피처 생성까지. 가설 검정·시각화·세그먼트 비교는 04에서 한다.

---

## 피처 목록

| 그룹 | 피처 |
|---|---|
| 프로필 | User_ID, 가입일자, 가입경로, 기기, 알림수신동의여부, signup_during_outage |
| 활동량 | total_events, active_days, last_active_date, routine_types_used |
| 온보딩 | onboard_completed, days_to_onboard |
| 초기경험 | active_days_7d, routine_count_7d, challenge_join_7d, challenge_explore_7d |
| 알림 | notif_received, notif_opened, notif_open_rate, notif_ad_count, consent_revoked |
| 플래그 | is_zero_event, d30_window_hit_outage |
| 타깃 | retained_d30, retained_d30_onboard |

In [1]:
import pandas as pd
import numpy as np

PROCESSED     = '../data/processed'
OUTAGE_START  = pd.Timestamp('2025-03-10')
OUTAGE_END    = pd.Timestamp('2025-03-14')
ROUTINE_TYPES = ['수면기록', '운동기록', '마음챙김', '식단기록']

events  = pd.read_csv(f'{PROCESSED}/events_clean.csv',  encoding='utf-8-sig', parse_dates=['Event_Time'])
profile = pd.read_csv(f'{PROCESSED}/profile_clean.csv', encoding='utf-8-sig',
                      parse_dates=['가입일자', '알림수신동의_변경일자'])

events['event_date'] = events['Event_Time'].dt.normalize()
ev = events

print(f'events : {len(ev):,}행  |  profile: {len(profile):,}행')

events : 1,734,134행  |  profile: 12,500행


---
## 1. 활동량 피처

In [2]:
activity = ev.groupby('User_ID').agg(
    total_events    =('Event_Type',  'count'),
    active_days     =('event_date',  'nunique'),
    last_active_date=('event_date',  'max'),
).reset_index()

routine_used = (
    ev[ev['Event_Type'].isin(ROUTINE_TYPES)]
    .groupby('User_ID')['Event_Type']
    .nunique()
    .rename('routine_types_used')
    .reset_index()
)

print(f'activity users: {len(activity):,}  routine users: {len(routine_used):,}')
activity[['total_events', 'active_days']].describe(percentiles=[.25,.5,.75,.95])

activity users: 12,453  routine users: 12,185


,total_events,active_days
count,12453.000000,12453.000000
mean,139.254316,24.834498
std,169.907910,28.588648
min,1.000000,1.000000
25%,20.000000,3.000000
50%,51.000000,9.000000
75%,217.000000,54.000000
95%,516.000000,79.000000
max,689.000000,89.000000


---
## 2. 온보딩 피처

In [3]:
ob_ev = (
    ev[ev['Event_Type'] == '온보딩_완료'][['User_ID', 'Event_Time']]
    .sort_values('Event_Time')
    .groupby('User_ID').first()
    .reset_index()
    .merge(profile[['User_ID', '가입일자']], on='User_ID')
)
ob_ev['onboard_completed'] = True
ob_ev['days_to_onboard']   = (ob_ev['Event_Time'].dt.normalize() - ob_ev['가입일자']).dt.days

print(f'온보딩 완료 유저: {len(ob_ev):,}  ({len(ob_ev)/len(profile)*100:.1f}%)')
ob_ev['days_to_onboard'].value_counts().head()

온보딩 완료 유저: 5,719  (45.8%)


days_to_onboard
0    5719
Name: count, dtype: int64

---
## 3. 초기 경험 피처 (첫 7일, day 0~6)

In [4]:
early = ev.merge(profile[['User_ID', '가입일자']], on='User_ID')
early['day_n'] = (early['event_date'] - early['가입일자']).dt.days
w7 = early[early['day_n'].between(0, 6)]

early_act = w7.groupby('User_ID').agg(
    active_days_7d  =('event_date',  'nunique'),
    routine_count_7d=('Event_Type',  lambda x: x.isin(ROUTINE_TYPES).sum()),
).reset_index()

challenge_join_7d = (
    w7[w7['Event_Type'] == '챌린지참여']['User_ID']
    .drop_duplicates().to_frame()
    .assign(challenge_join_7d=True)
)
challenge_explore_7d = (
    w7[w7['Event_Type'] == '챌린지_탐색']['User_ID']
    .drop_duplicates().to_frame()
    .assign(challenge_explore_7d=True)
)

print(f'7일 활동 유저     : {len(early_act):,}')
print(f'challenge_join_7d : {len(challenge_join_7d):,}  ({len(challenge_join_7d)/len(profile)*100:.1f}%)')
print(f'challenge_explore : {len(challenge_explore_7d):,}  ({len(challenge_explore_7d)/len(profile)*100:.1f}%)')
early_act[['active_days_7d', 'routine_count_7d']].describe()

7일 활동 유저     : 12,453
challenge_join_7d : 8,185  (65.5%)
challenge_explore : 7,872  (63.0%)


,active_days_7d,routine_count_7d
count,12453.000000,12453.000000
mean,5.209106,12.706015
std,2.354870,8.982032
min,1.000000,0.000000
25%,3.000000,6.000000
50%,7.000000,11.000000
75%,7.000000,18.000000
max,7.000000,64.000000


---
## 4. 알림 피처

In [5]:
nr = ev[ev['Event_Type'] == '알림수신'].groupby('User_ID').size().rename('notif_received').reset_index()
no = ev[ev['Event_Type'] == '알림오픈' ].groupby('User_ID').size().rename('notif_opened').reset_index()
na = (
    ev[(ev['Event_Type'] == '알림수신') & (ev['알림_유형'] == '광고성')]
    .groupby('User_ID').size().rename('notif_ad_count').reset_index()
)

notif = nr.merge(no, on='User_ID', how='outer').merge(na, on='User_ID', how='outer')
notif['notif_open_rate'] = notif['notif_opened'] / notif['notif_received'].replace(0, np.nan)

print(f'알림 수신 유저: {len(notif):,}')
notif[['notif_received', 'notif_opened', 'notif_open_rate', 'notif_ad_count']].describe()

알림 수신 유저: 8,156


,notif_received,notif_opened,notif_open_rate,notif_ad_count
count,8156.000000,5127.000000,5127.000000,6079.000000
mean,24.235287,4.138678,0.188812,12.528541
std,28.454663,3.461521,0.187274,12.998258
min,1.000000,1.000000,0.012500,1.000000
25%,3.000000,1.000000,0.090398,2.000000
50%,8.000000,3.000000,0.128205,5.000000
75%,53.000000,6.000000,0.200000,23.000000
max,89.000000,20.000000,1.000000,54.000000


---
## 5. D30 리텐션 타깃

In [6]:
app = ev[ev['Event_Type'] == '앱실행'][['User_ID', 'event_date']].merge(
    profile[['User_ID', '가입일자']], on='User_ID'
)
app['day_n'] = (app['event_date'] - app['가입일자']).dt.days

# 28~30일 창: 단일 날 D30보다 ±1~2일 측정 오차에 강건, 업계 표준 버퍼
r30 = (
    app[app['day_n'].between(28, 30)]['User_ID']
    .drop_duplicates().to_frame()
    .assign(retained_d30=True)
)

# retained_d30_onboard: Day 0 = 온보딩 완료일 (완료자에게만 의미 있음)
app_ob = ev[ev['Event_Type'] == '앱실행'][['User_ID', 'event_date']].merge(
    ob_ev[['User_ID', 'Event_Time']].rename(columns={'Event_Time': 'onboard_time'}),
    on='User_ID'
)
app_ob['day_n_ob'] = (app_ob['event_date'] - app_ob['onboard_time'].dt.normalize()).dt.days
r30_ob = (
    app_ob[app_ob['day_n_ob'].between(28, 30)]['User_ID']
    .drop_duplicates().to_frame()
    .assign(retained_d30_onboard=True)
)

print(f'retained_d30         : {len(r30):,}  ({len(r30)/len(profile)*100:.1f}%)')
print(f'retained_d30_onboard : {len(r30_ob):,}  ({len(r30_ob)/len(ob_ev)*100:.1f}% of 온보딩 완료자)')

retained_d30         : 3,984  (31.9%)
retained_d30_onboard : 2,222  (38.9% of 온보딩 완료자)


---
## 6. 조립 — user_features 테이블

In [7]:
uft = profile[[
    'User_ID', '가입일자', '가입경로', '기기', '알림수신동의여부',
    '알림수신동의_변경일자', 'signup_during_outage'
]].copy()

uft = (
    uft
    .merge(activity,                                               on='User_ID', how='left')
    .merge(routine_used,                                           on='User_ID', how='left')
    .merge(ob_ev[['User_ID', 'onboard_completed', 'days_to_onboard']], on='User_ID', how='left')
    .merge(early_act,                                              on='User_ID', how='left')
    .merge(challenge_join_7d,                                      on='User_ID', how='left')
    .merge(challenge_explore_7d,                                   on='User_ID', how='left')
    .merge(notif[['User_ID', 'notif_received', 'notif_opened',
                  'notif_open_rate', 'notif_ad_count']],           on='User_ID', how='left')
    .merge(r30,                                                    on='User_ID', how='left')
    .merge(r30_ob,                                                 on='User_ID', how='left')
)

# 정수형 피처: 이벤트 없는 유저는 0
int0_cols = ['total_events', 'active_days', 'routine_types_used',
             'active_days_7d', 'routine_count_7d',
             'notif_received', 'notif_opened', 'notif_ad_count']
uft[int0_cols] = uft[int0_cols].fillna(0).astype(int)

# 불리언 피처: 이벤트 없거나 해당 행동 없는 유저는 False
bool_false_cols = ['onboard_completed', 'challenge_join_7d', 'challenge_explore_7d', 'retained_d30']
uft[bool_false_cols] = uft[bool_false_cols].fillna(False)

# retained_d30_onboard: 온보딩 완료자는 True/False, 나머지는 NaN 유지
onboard_mask = uft['onboard_completed']
uft.loc[onboard_mask & uft['retained_d30_onboard'].isna(), 'retained_d30_onboard'] = False

# 파생 플래그
uft['is_zero_event']   = uft['total_events'] == 0
uft['consent_revoked'] = uft['알림수신동의_변경일자'].notna()

# d30_window_hit_outage: 활성 판정 창이 로그 수집 장애 구간과 겹치는 유저
d30_win_start = uft['가입일자'] + pd.Timedelta(days=28)
d30_win_end   = uft['가입일자'] + pd.Timedelta(days=30)
uft['d30_window_hit_outage'] = (d30_win_start <= OUTAGE_END) & (d30_win_end >= OUTAGE_START)

uft = uft.drop(columns=['알림수신동의_변경일자'])

print(f'shape: {uft.shape}')
uft.head(3)

shape: (12500, 25)


,User_ID,가입일자,가입경로,기기,알림수신동의여부,signup_during_outage,total_events,active_days,last_active_date,routine_types_used,...,challenge_explore_7d,notif_received,notif_opened,notif_open_rate,notif_ad_count,retained_d30,retained_d30_onboard,is_zero_event,consent_revoked,d30_window_hit_outage
0,U0000001,2025-01-25,오가닉,iOS,동의,False,512,80,2025-04-24,4,...,True,80,7,0.0875,28,True,True,False,False,False
1,U0000002,2025-05-06,오가닉,iOS,미동의,False,53,10,2025-05-16,4,...,True,0,0,NaN,0,False,NaN,False,True,False
2,U0000003,2025-05-14,오가닉,iOS,미동의,False,3,1,2025-05-14,1,...,False,0,0,NaN,0,False,NaN,False,False,False


---
## 7. 검증 & 요약

In [8]:
assert len(uft) == 12_500,                   f'행 수 오류: {len(uft)}'
assert uft['User_ID'].nunique() == 12_500,   'User_ID 중복 존재'
assert uft['total_events'].min() >= 0

print(f'shape          : {uft.shape}')
print(f'User_ID unique : {uft["User_ID"].nunique():,}')
print(f'is_zero_event  : {uft["is_zero_event"].sum():,}')
print()
print('=== D30 리텐션 (rough) ===')
n_r30    = uft['retained_d30'].sum()
n_r30_ob = uft['retained_d30_onboard'].sum()
n_ob     = uft['onboard_completed'].sum()
print(f'retained_d30         : {n_r30:,} / {len(uft):,}  = {n_r30/len(uft):.3f}')
print(f'retained_d30_onboard : {n_r30_ob:.0f} / {n_ob:,}  = {n_r30_ob/n_ob:.3f}  (온보딩 완료자 기준)')
print(f'd30_window_hit_outage: {uft["d30_window_hit_outage"].sum():,}명')
print()
print('=== 주요 피처 결측 요약 ===')
miss = uft.isnull().sum()
print(miss[miss > 0])

shape          : (12500, 25)
User_ID unique : 12,500
is_zero_event  : 47

=== D30 리텐션 (rough) ===
retained_d30         : 3,984 / 12,500  = 0.319
retained_d30_onboard : 2222 / 5,719  = 0.389  (온보딩 완료자 기준)
d30_window_hit_outage: 1,075명

=== 주요 피처 결측 요약 ===
last_active_date          47
days_to_onboard         6781
notif_open_rate         7373
retained_d30_onboard    6781
dtype: int64


In [9]:
print('=== 최종 컬럼 목록 ===')
for i, c in enumerate(uft.columns, 1):
    print(f'  {i:2d}. {c}')

=== 최종 컬럼 목록 ===
   1. User_ID
   2. 가입일자
   3. 가입경로
   4. 기기
   5. 알림수신동의여부
   6. signup_during_outage
   7. total_events
   8. active_days
   9. last_active_date
  10. routine_types_used
  11. onboard_completed
  12. days_to_onboard
  13. active_days_7d
  14. routine_count_7d
  15. challenge_join_7d
  16. challenge_explore_7d
  17. notif_received
  18. notif_opened
  19. notif_open_rate
  20. notif_ad_count
  21. retained_d30
  22. retained_d30_onboard
  23. is_zero_event
  24. consent_revoked
  25. d30_window_hit_outage


---
## 8. 저장

In [10]:
uft.to_csv(f'{PROCESSED}/user_features.csv', index=False, encoding='utf-8-sig')
print(f'저장 완료: {PROCESSED}/user_features.csv  ({len(uft):,}행 × {uft.shape[1]}열)')

저장 완료: ../data/processed/user_features.csv  (12,500행 × 25열)
